# 07 — Sentence Trajectories v2

**Engineering question**

How do sentence tokens trace paths through transformer representation space?

Notebook 00 defined the conceptual object:

\[
x_i^\ell
\]

the hidden representation of token \(i\) at layer \(\ell\).

This notebook extracts real hidden states from a pretrained GPT-style model and improves the trajectory visualization by:

```text
raw hidden states
→ sentence-centered trajectories
→ first-token-relative trajectories
→ PCA shape projections
→ local step alignment
```

Curvature itself is reserved for Notebook 13. This notebook prepares the measured trajectory objects and the adjacent-step alignment quantities that curvature will aggregate.


## Setup

The notebook can run from the repo root or inside `notebooks/`.

It creates:

```text
figures/
results/csv/
results/json/
results/arrays/
results/07_outputs.zip
```


In [ ]:
from pathlib import Path
import json
import zipfile
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"
ARR = RES / "arrays"

for p in [FIG, RES, CSV, JS, ARR]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Load model and tokenizer

Start with `distilgpt2`.

The aim is not model quality. The aim is to validate the hidden-state trajectory extraction pipeline quickly.

Once the pipeline is stable, switch to `gpt2` or larger GPT-2 variants.


In [ ]:
MODEL_NAME = "distilgpt2"

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, output_hidden_states=True)
    model.to(DEVICE)
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("model:", MODEL_NAME)
    print("device:", DEVICE)
    print("num transformer layers:", getattr(model.config, "n_layer", "unknown"))
    print("hidden size:", getattr(model.config, "n_embd", "unknown"))

except Exception as e:
    print("Model setup failed.")
    print("Install dependencies with: pip install -r requirements.txt")
    raise e


## 2. Sentence batch

Use a compact batch that includes the lab-report title, simple language, and geometry language.

The batch provides enough variation to check that the trajectory extraction pipeline works across multiple inputs.


In [ ]:
sentences = [
    "Trajectory straightening specifies language prediction.",
    "The cat sat on the mat.",
    "Large language models predict the next token.",
    "Geometry organizes predictive representations.",
    "Curvature decreases across transformer depth.",
]

batch = tokenizer(
    sentences,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

batch = {k: v.to(DEVICE) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch, output_hidden_states=True)

hidden_states = outputs.hidden_states
hidden = torch.stack(hidden_states, dim=0).detach().cpu().numpy()

input_ids = batch["input_ids"].detach().cpu().numpy()
attention_mask = batch["attention_mask"].detach().cpu().numpy()

tokens_by_sentence = [
    tokenizer.convert_ids_to_tokens(row)
    for row in input_ids
]

print("hidden tensor shape:", hidden.shape)
print("meaning: [layers including embeddings, batch, tokens, hidden_dim]")
print("input_ids shape:", input_ids.shape)


## 3. Hidden-state tensor

The core object is:

```text
layers × sentences × tokens × hidden_dimension
```

This tensor is saved for Notebook 13.


In [ ]:
shape_record = {
    "model": MODEL_NAME,
    "hidden_shape": list(hidden.shape),
    "input_ids_shape": list(input_ids.shape),
    "num_hidden_layers_including_embedding": int(hidden.shape[0]),
    "batch_size": int(hidden.shape[1]),
    "sequence_length": int(hidden.shape[2]),
    "hidden_dim": int(hidden.shape[3]),
}

with open(JS / "07_hidden_state_shape.json", "w") as f:
    json.dump(shape_record, f, indent=2)

shape_df = pd.DataFrame(
    [
        {"axis": 0, "name": "layers including embedding", "size": hidden.shape[0]},
        {"axis": 1, "name": "sentences", "size": hidden.shape[1]},
        {"axis": 2, "name": "tokens", "size": hidden.shape[2]},
        {"axis": 3, "name": "hidden dimension", "size": hidden.shape[3]},
    ]
)

shape_df.to_csv(CSV / "07_hidden_state_shape.csv", index=False)
shape_df


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))

labels = ["layers", "sentences", "tokens", "hidden dim"]
sizes = shape_df["size"].tolist()
x = np.arange(len(labels))

ax.bar(x, sizes)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("size")
ax.set_title("Hidden-State Tensor Shape")
ax.grid(True, axis="y", alpha=0.25)

for xi, size in zip(x, sizes):
    ax.text(xi, size, str(size), ha="center", va="bottom", fontsize=10)

savefig(fig, "07_hidden_state_shapes.png")
plt.show()


## 4. Token metadata

Store tokenized sentences.

Each active token corresponds to one position in the trajectory.


In [ ]:
def clean_token(token):
    return token.replace("Ġ", " ").replace("Ċ", "\\n")

rows = []
for s_idx, sentence in enumerate(sentences):
    for t_idx, token in enumerate(tokens_by_sentence[s_idx]):
        active = int(attention_mask[s_idx, t_idx])
        rows.append(
            {
                "sentence_index": s_idx,
                "sentence": sentence,
                "token_index": t_idx,
                "token": token,
                "clean_token": clean_token(token),
                "active": active,
            }
        )

token_df = pd.DataFrame(rows)
token_df.to_csv(CSV / "07_sentence_metadata.csv", index=False)
token_df.head(30)


In [ ]:
counts = token_df[token_df["active"] == 1].groupby("sentence_index").size()

fig, ax = plt.subplots(figsize=(8.8, 4.2))
ax.bar(counts.index.astype(str), counts.values)
ax.set_title("Active Token Count Per Sentence")
ax.set_xlabel("sentence index")
ax.set_ylabel("active tokens")
ax.grid(True, axis="y", alpha=0.25)

for xi, val in enumerate(counts.values):
    ax.text(xi, val, str(int(val)), ha="center", va="bottom")

savefig(fig, "07_token_counts.png")
plt.show()


## 5. Trajectory helpers

Notebook 07 uses three trajectory views:

1. **raw** hidden states,
2. **centered** hidden states, subtracting the sentence mean,
3. **origin-relative** hidden states, subtracting the first token.

The centered and origin-relative views make the shape easier to see.

Notebook 13 should compute curvature in hidden space, not from the 2D PCA figure.


In [ ]:
from sklearn.decomposition import PCA

def active_hidden_for_sentence(sentence_index, layer_index):
    mask = attention_mask[sentence_index].astype(bool)
    return hidden[layer_index, sentence_index, mask, :]

def center_points(points):
    return points - points.mean(axis=0, keepdims=True)

def origin_relative_points(points):
    return points - points[:1, :]

def pca_project(points, n_components=2, mode="centered"):
    if points.shape[0] < 2:
        raise ValueError("Need at least two tokens to project a trajectory.")

    if mode == "centered":
        work = center_points(points)
    elif mode == "origin":
        work = origin_relative_points(points)
    elif mode == "raw":
        work = points
    else:
        raise ValueError("mode must be one of: raw, centered, origin")

    pca = PCA(n_components=n_components)
    xy = pca.fit_transform(work)
    return xy, pca

def step_vectors(points):
    return np.diff(points, axis=0)

def step_lengths(points):
    return np.linalg.norm(step_vectors(points), axis=1)

def adjacent_step_cosines(points, eps=1e-12):
    v = step_vectors(points)
    if len(v) < 2:
        return np.array([])
    a = v[:-1]
    b = v[1:]
    denom = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    denom = np.maximum(denom, eps)
    return np.sum(a * b, axis=1) / denom

def annotate_token_indices(ax, xy, fontsize=8):
    for i, (x, y) in enumerate(xy):
        ax.text(x, y, str(i), ha="center", va="center", fontsize=fontsize)


## 6. Raw versus centered PCA

The first version of Notebook 07 showed that raw PCA can be dominated by one large direction.

This comparison shows why centering matters.

Centered PCA better emphasizes the trajectory shape.


In [ ]:
sentence_index = 0
layer_index = min(3, hidden.shape[0] - 1)
points = active_hidden_for_sentence(sentence_index, layer_index)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))

for ax, mode in zip(axes, ["raw", "centered", "origin"]):
    xy, pca = pca_project(points, mode=mode)
    ax.plot(xy[:, 0], xy[:, 1], marker="o", linewidth=2.0)
    ax.scatter(xy[0, 0], xy[0, 1], s=80, marker="s", label="start")
    ax.scatter(xy[-1, 0], xy[-1, 1], s=90, marker="*", label="end")
    annotate_token_indices(ax, xy, fontsize=8)
    ax.set_title(f"{mode} PCA\\nvar={pca.explained_variance_ratio_.sum():.2f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.25)

axes[0].legend()
fig.suptitle("Trajectory Projection: Raw vs Centered vs Origin-Relative", fontsize=16, y=1.04)

savefig(fig, "07_projection_modes.png")
plt.show()


## 7. One sentence across selected layers

Use centered PCA to view one sentence trajectory across several layers.

The labels mark token index along the path.


In [ ]:
sentence_index = 0
selected_layers = [0, 1, 2, 4, 6]
max_layer = hidden.shape[0] - 1
selected_layers = [min(layer, max_layer) for layer in selected_layers]

fig, axes = plt.subplots(1, len(selected_layers), figsize=(4.0 * len(selected_layers), 3.8))

if len(selected_layers) == 1:
    axes = [axes]

for ax, layer_index in zip(axes, selected_layers):
    points = active_hidden_for_sentence(sentence_index, layer_index)
    xy, pca = pca_project(points, mode="centered")

    ax.plot(xy[:, 0], xy[:, 1], marker="o", linewidth=2.0)
    ax.scatter(xy[0, 0], xy[0, 1], s=80, marker="s", label="start")
    ax.scatter(xy[-1, 0], xy[-1, 1], s=90, marker="*", label="end")
    annotate_token_indices(ax, xy, fontsize=8)

    ax.set_title(f"Layer {layer_index}\\nPCA var {pca.explained_variance_ratio_.sum():.2f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.25)

axes[0].legend(loc="best")
fig.suptitle("Centered Sentence Trajectory Across Selected Layers", fontsize=16, y=1.03)

savefig(fig, "07_sentence_trajectory_selected_layers.png")
plt.show()


## 8. Layer projection grid

This grid is the main visual output of Notebook 07.

It shows a single sentence trajectory across transformer depth using centered PCA.


In [ ]:
sentence_index = 0
num_layers_total = hidden.shape[0]

grid_layers = np.linspace(0, num_layers_total - 1, min(num_layers_total, 8), dtype=int)
ncols = 4
nrows = math.ceil(len(grid_layers) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.6 * nrows))
axes = np.array(axes).reshape(-1)

for ax, layer_index in zip(axes, grid_layers):
    points = active_hidden_for_sentence(sentence_index, layer_index)
    xy, pca = pca_project(points, mode="centered")

    ax.plot(xy[:, 0], xy[:, 1], marker="o", linewidth=1.8)
    ax.scatter(xy[0, 0], xy[0, 1], s=65, marker="s")
    ax.scatter(xy[-1, 0], xy[-1, 1], s=90, marker="*")
    annotate_token_indices(ax, xy, fontsize=7)
    ax.set_title(f"Layer {layer_index}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(True, alpha=0.2)

for ax in axes[len(grid_layers):]:
    ax.axis("off")

fig.suptitle("Layerwise Token Trajectory Projection, Centered PCA", fontsize=16, y=1.02)

savefig(fig, "07_layer_projection_grid.png")
plt.show()


## 9. Local step alignment

Curvature is based on the angle between adjacent displacement vectors.

Notebook 07 computes the cosine alignment first:

\[
\cos_i^\ell =
\frac{v_i^\ell \cdot v_{i+1}^\ell}
{\|v_i^\ell\| \, \|v_{i+1}^\ell\|}
\]

Higher cosine means adjacent steps point in more similar directions.

Notebook 13 will convert this into angle/curvature.


In [ ]:
records = []

for s_idx, sentence in enumerate(sentences):
    for layer_index in range(hidden.shape[0]):
        points = active_hidden_for_sentence(s_idx, layer_index)
        points_centered = center_points(points)
        points_origin = origin_relative_points(points)

        steps = step_lengths(points)
        cosines = adjacent_step_cosines(points)

        xy, pca = pca_project(points, mode="centered")

        records.append(
            {
                "sentence_index": s_idx,
                "sentence": sentence,
                "layer": layer_index,
                "active_tokens": int(points.shape[0]),
                "trajectory_length": float(steps.sum()),
                "mean_step_length": float(steps.mean()) if len(steps) else np.nan,
                "mean_adjacent_step_cosine": float(cosines.mean()) if len(cosines) else np.nan,
                "median_adjacent_step_cosine": float(np.median(cosines)) if len(cosines) else np.nan,
                "pca_2d_variance_centered": float(pca.explained_variance_ratio_.sum()),
            }
        )

stats = pd.DataFrame(records)
stats.to_csv(CSV / "07_layer_statistics.csv", index=False)
stats.to_json(JS / "07_layer_statistics.json", orient="records", indent=2)

stats.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.8))

for s_idx, sentence in enumerate(sentences):
    subset = stats[stats["sentence_index"] == s_idx]
    ax.plot(
        subset["layer"],
        subset["mean_adjacent_step_cosine"],
        marker="o",
        markersize=3,
        label=f"sentence {s_idx}",
    )

ax.axhline(0, linewidth=1)
ax.set_title("Mean Adjacent-Step Cosine Across Layers")
ax.set_xlabel("layer")
ax.set_ylabel("mean adjacent-step cosine")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)

savefig(fig, "07_adjacent_step_cosine.png")
plt.show()


## 10. Trajectory summary

Keep trajectory length as a sanity check, but do not over-interpret it.

Curvature and adjacent-step alignment are more directly connected to the paper.


In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.8))

for s_idx, sentence in enumerate(sentences):
    subset = stats[stats["sentence_index"] == s_idx]
    ax.plot(
        subset["layer"],
        subset["trajectory_length"],
        marker="o",
        markersize=3,
        label=f"sentence {s_idx}",
    )

ax.set_title("Trajectory Length Across Layers")
ax.set_xlabel("layer")
ax.set_ylabel("trajectory length in hidden-state space")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)

savefig(fig, "07_trajectory_length.png")
plt.show()


## 11. Save arrays

Save hidden states, inputs, and masks.

Notebook 13 can reload these outputs and compute curvature without re-running model inference.


In [ ]:
npz_path = ARR / "07_hidden_states.npz"

np.savez_compressed(
    npz_path,
    hidden=hidden,
    input_ids=input_ids,
    attention_mask=attention_mask,
    sentences=np.array(sentences, dtype=object),
)

print("saved:", npz_path)
print("hidden:", hidden.shape)


## 12. Compact interpretation

This notebook validates the first measured object in the repo:

\[
x_i^\ell
\]

A sentence is not represented by one vector. It becomes a sequence of token-indexed hidden states across layers.

That sequence forms a trajectory.

Notebook 07 also prepares the adjacent-step cosine:

\[
\cos_i^\ell =
\frac{v_i^\ell \cdot v_{i+1}^\ell}
{\|v_i^\ell\| \, \|v_{i+1}^\ell\|}
\]

Notebook 13 should convert this into the paper's curvature metric.


## 13. Export and download

Package all Notebook 07 outputs.


In [ ]:
zip_path = RES / "07_outputs.zip"

files_to_zip = [
    FIG / "07_hidden_state_shapes.png",
    FIG / "07_token_counts.png",
    FIG / "07_projection_modes.png",
    FIG / "07_sentence_trajectory_selected_layers.png",
    FIG / "07_layer_projection_grid.png",
    FIG / "07_adjacent_step_cosine.png",
    FIG / "07_trajectory_length.png",
    CSV / "07_hidden_state_shape.csv",
    CSV / "07_sentence_metadata.csv",
    CSV / "07_layer_statistics.csv",
    JS / "07_hidden_state_shape.json",
    JS / "07_layer_statistics.json",
    ARR / "07_hidden_states.npz",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

stats.head()


## Next notebook

**13 — Curvature**

The next notebook should compute the curvature metric from the saved hidden-state tensor.

Candidate outputs:

```text
13_curvature_by_layer.png
13_curvature_heatmap.png
13_alignment_vs_curvature.png
13_curvature_summary.csv
```

Core question:

```text
Do sentence trajectories become straighter from early to middle transformer layers?
```
